# CRISP-DM Phase 2 — Data Understanding | Kaggle Apple-Watch Dataset

This notebook explores the **current** training source: the **Kaggle "Gym Workout IMU
Dataset"**, recorded on an **Apple Watch SE** (left wrist, 100 Hz). It replaces the
earlier RecoFit/MM-Fit exploration, which were abandoned for device-domain mismatch.

It shows: the file layout, columns and units, the 21 exercise abbreviations and their
full names, the **3 chosen target classes** and their set counts, the 100 Hz sampling
rate, the single-subject limitation, and example signals.

All heavy logic is delegated to `src/ml4b/` (e.g. `ml4b.data.kaggle_loader`). See
**ADR-016** (class choice) and `docs/data_understanding/dataset_evaluation.md`.

> **Dataset required:** `data/raw/kaggle_gym_imu/` (download link in the Setup guides).
> If it is missing the data cells will raise a clear error; the markdown still documents
> the findings.

In [ ]:
# Imports and project paths (no hardcoded paths — resolved via config).
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from ml4b.utils.config import DATA_RAW
from ml4b.data.kaggle_loader import ABBREV_TO_CLASS, TARGET_CLASSES, load_kaggle_file

KAGGLE_DIR = DATA_RAW / 'kaggle_gym_imu'
print('Kaggle dir:', KAGGLE_DIR)
print('Exists   :', KAGGLE_DIR.exists())
print('Target classes:', TARGET_CLASSES)

## 1. Files and layout
The dataset is a **flat folder of single-set CSV files**. Each filename encodes the
exercise: `DDMMYY_ABBREV_Wweight_Sset_Rreps-timestamp.csv`.

In [ ]:
csv_files = sorted(KAGGLE_DIR.glob('*.csv'))
print(f'CSV files: {len(csv_files)}')
for p in csv_files[:5]:
    print(' ', p.name)

## 2. Columns and units (Apple CoreMotion)
Each file holds CoreMotion `wristMotion_*` channels. Acceleration components are in
**g**, rotation rate in **rad/s** — the same conventions as Sensor Logger, which is
why this dataset matches the deployment device (ADR-016).

In [ ]:
example = pd.read_csv(csv_files[0])
print('Columns:', list(example.columns))
print('Rows in example file:', len(example))
# Schema consistency across all files (cheap header-only check).
same = all(list(pd.read_csv(p, nrows=0).columns) == list(example.columns) for p in csv_files)
print('All files share the same columns:', same)

## 3. Exercise abbreviations -> full names
The dataset contains **21** distinct exercises (abbreviations). The full-name decode
below comes from the dataset's published description; the three highlighted groups are
our target classes (ADR-016).

In [ ]:
ABBREV_FULL = {
    '30BP': '30 deg (Barbell) Bench Press', '30DBP': '30 deg Incline Dumbbell Bench Press',
    '45DBP': '45 deg Incline Dumbbell Bench Press',
    'AIDBC': 'Alternating Incline Dumbbell Bicep Curl', 'IDBC': 'Incline Dumbbell Bicep Curl',
    'PREC': 'Machine Preacher Curl',
    'CGOCTE': 'Close-Grip Overhead Cable Triceps Extension', 'MTE': 'Machine Triceps Extension',
    'SAOCTE': 'Single-Arm Overhead Cable Triceps Extension',
    'SAODTE': 'Single-Arm Overhead Dumbbell Triceps Extension',
    'CGCR': 'Close-Grip Cable Row', 'NGCR': 'Neutral-Grip Cable Row', 'MGTBR': 'Machine/Mid-Grip T-Bar Row',
    'APULL': 'Assisted Pull-up', 'SBLP': 'Straight-Bar Lat Pulldown',
    'DLR': 'Dumbbell Lateral Raise', 'SACLR': 'Single-Arm Cable Lateral Raise',
    'DSP': 'Dumbbell Shoulder Press', 'MSP': 'Machine Shoulder Press',
    'MIBP': 'Machine Incline Bench Press', 'DWC': 'Dumbbell Wrist Curl',
}
# Count sets per abbreviation from the filenames (one file == one set).
from collections import Counter
sets_per_abbrev = Counter(p.name.split('_')[1] for p in csv_files)
rows = [
    {'abbrev': a, 'full_name': ABBREV_FULL.get(a, '?'),
     'sets': sets_per_abbrev[a], 'target_class': ABBREV_TO_CLASS.get(a, '-')}
    for a in sorted(sets_per_abbrev)
]
abbrev_df = pd.DataFrame(rows).sort_values(['target_class', 'abbrev'])
print(f'Distinct abbreviations: {len(abbrev_df)}  |  total sets: {sum(sets_per_abbrev.values())}')
abbrev_df.reset_index(drop=True)

## 4. The 3 target classes and their set counts (ADR-016)
We group biomechanically-equivalent abbreviations into three classes spanning three
distinct movement axes: **bicep_curl** (elbow flexion), **tricep_extension** (overhead
extension), **row** (horizontal pull). `push_up` is **absent** from this dataset
(`APULL` is an assisted *pull*-up), so it was dropped.

In [ ]:
target = abbrev_df[abbrev_df['target_class'] != '-']
sets_per_class = target.groupby('target_class')['sets'].sum().reindex(sorted(TARGET_CLASSES))
print('Sets per target class:')
print(sets_per_class.to_string())
print('Total target sets:', int(sets_per_class.sum()))

fig, ax = plt.subplots(figsize=(6, 3.5))
sets_per_class.plot(kind='bar', color=['#4C78A8', '#54A24B', '#E45756'], ax=ax)
ax.set_ylabel('number of sets'); ax.set_title('Sets per target class (Kaggle)')
ax.set_xlabel(''); plt.xticks(rotation=0); plt.tight_layout(); plt.show()

## 5. Sampling rate (confirm 100 Hz)
Computed from `secondsElapsed` deltas across a sample of files.

In [ ]:
rates = []
for p in csv_files[:15]:
    s = pd.read_csv(p, usecols=['secondsElapsed'])['secondsElapsed'].diff().dropna()
    if s.median() and s.median() > 0:
        rates.append(1.0 / s.median())
print(f'Sampling rate over {len(rates)} files: '
      f'min={min(rates):.1f} Hz, max={max(rates):.1f} Hz, mean={np.mean(rates):.1f} Hz')

## 6. Single-subject limitation and sensor lag
There is **no subject/athlete column** — the dataset documents one collector on one
Apple Watch, so it is effectively **single subject**. This drives the leave-one-set-out
evaluation (ADR-021). Each file also begins with ~0.24 s of CoreMotion warm-up lag
(all-NaN rows), which the loader trims.

In [ ]:
lag = example[example['wristMotion_accelerationX'].isna()]
print('Leading NaN (warm-up lag) rows in example file:', len(lag))
print('Lag duration (s):', round(float(example.loc[lag.index, 'secondsElapsed'].max() or 0), 3))
print('Subject/athlete column present:',
      any('subject' in c.lower() or 'athlete' in c.lower() for c in example.columns))

## 7. Example signals (one bicep-curl set)
Loaded through the **current loader** (`load_kaggle_file`), which trims the lag and
reconstructs total acceleration in g + gyro in rad/s — the canonical representation
shared with the app.

In [ ]:
# Pick the first file that maps to bicep_curl.
bicep_file = next(p for p in csv_files if ABBREV_TO_CLASS.get(p.name.split('_')[1]) == 'bicep_curl')
df = load_kaggle_file(bicep_file)
accel_mag = np.sqrt(df['ax']**2 + df['ay']**2 + df['az']**2)
gyro_mag = np.sqrt(df['gx']**2 + df['gy']**2 + df['gz']**2)

fig, axs = plt.subplots(2, 1, figsize=(9, 4.5), sharex=True)
axs[0].plot(df['timestamp'], accel_mag, lw=0.8); axs[0].set_ylabel('|accel| (g)')
axs[0].set_title(f'Bicep curl set: {bicep_file.name}')
axs[1].plot(df['timestamp'], gyro_mag, lw=0.8, color='#E45756'); axs[1].set_ylabel('|gyro| (rad/s)')
axs[1].set_xlabel('time (s)'); plt.tight_layout(); plt.show()

## 8. Takeaways
- **Device match:** Apple Watch, 100 Hz — same as deployment (ADR-016).
- **3 target classes:** bicep_curl (24 sets), tricep_extension (30), row (21).
- **Single subject** -> evaluate with **leave-one-set-out** CV, not a random split (ADR-021).
- Canonical units: total acceleration in g, gyro in rad/s (see `ml4b.data.canonical`).

Next: **03_data_preparation** turns these recordings into windowed, feature-extracted,
augmented training data.